# 04 · Recursión sobre listas (divide y vencerás) y `foldRight`

> **En el examen:** suele ser el **Ejercicio 3** (3-4 puntos), y a veces también parte del
> 4. El formato casi siempre es el mismo: te dan una función con sus tests y te piden
> implementarla **varias veces**, de formas distintas:
> - **recursión (no final)** ← este archivo
> - **`foldRight`** ← este archivo
> - recursión **final** y **`foldLeft`** ← archivo 05
> - con **`map` / `filter` / `flatMap`…** ← archivo 06
>
> Sale en prácticamente **todos** los exámenes. Es lo que más puntos vale en total.

Necesitas del archivo 01: listas (`Nil`, `::`), `match`, tuplas, `Option`, `Either`,
genéricos y lambdas.

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. La idea: "reduce y vencerás"

Una lista es **o** vacía (`Nil`) **o** una cabeza con un resto (`head :: tail`). Para
resolver un problema sobre una lista:

1. **Caso base** (lista vacía): ¿cuál es la respuesta para `Nil`? Suele ser obvio.
2. **Caso recursivo** (`head :: tail`): **supón que ya tienes la solución para el resto**
   (`tailSol`) y pregúntate: *¿cómo la combino con la cabeza para obtener la solución de la
   lista entera?*

No intentes "seguir" la recursión mentalmente hasta el final. Confía en que `f(tail)` te da
la respuesta correcta del resto y solo piensa en **un paso**.

### La plantilla (viene en el preámbulo de algunos exámenes)

In [ ]:
def divideAndConquer[A, B](l: List[A]): B =
    l match
        case Nil => ???                              // solución directa
        case head :: tail =>
            val tailSol: B = divideAndConquer(tail)  // solución del resto
            ???  /* combinar head y tailSol */

### Ejemplo mínimo: sumar

In [ ]:
def suma(l: List[Int]): Int =
    l match
        case Nil => 0
        case head :: tail =>
            val tailSol = suma(tail)    // suma del resto
            head + tailSol              // combinar

Para `List(5, 2, 1)`: `tailSol` sería `3` (la suma de `List(2, 1)`), y devolvemos `5 + 3`.

**Pregúntate siempre con un ejemplo de los tests**: "si la lista es `List(x, ...resto)` y
ya sé la respuesta del resto, ¿qué devuelvo?"

---

## 2. Ejemplo guiado: `collect` (25-26 Extraordinaria)

Enunciado: `collect(f, l)` aplica `f: A => Option[B]` a cada elemento y se queda con los
`b` de los `Some(b)`, en orden.

Tests:

In [ ]:
val f: Int => Option[String] = n => if n % 2 == 0 then Some(s"par($n)") else None
collect(f, List(1, 2, 3, 4)) shouldBe List("par(2)", "par(4)")
collect(f, List()) shouldBe List()

Razonamiento:
- Base: lista vacía → no hay nada que recoger → `Nil`.
- Recursivo: tengo `head` y `tailSol = collect(f, tail)` (lo recogido del resto).
  - si `f(head)` es `Some(b)` → pongo `b` delante: `b :: tailSol`
  - si es `None` → `tailSol` tal cual

In [ ]:
def collect[A, B](f: A => Option[B], l: List[A]): List[B] =
    l match
        case Nil => Nil
        case head :: tail =>
            f(head) match
                case Some(b) => b :: collect(f, tail)
                case None    => collect(f, tail)

---

## 3. `foldRight`: la recursión ya escrita

Fíjate en que todas estas funciones tienen la misma estructura: un valor para `Nil` y una
forma de combinar `head` con `tailSol`. `foldRight` es una **función de orden superior**
(una función que recibe otra función) que hace la recursión por ti. Tú solo das las dos
piezas que cambian.

In [ ]:
def foldRight[B](nil: B)(cons: (A, B) => B): B
//               ^^^^^^   ^^^^^^^^^^^^^^^^^^
//               caso     cómo combinar (head, tailSol)
//               base

Se usa como método de la lista: `l.foldRight(valorBase)(funcionCombinar)`.

Qué hace por dentro, para `List(1, 2, 3)`:

```
List(1, 2, 3).foldRight(z)(f)  ==  f(1, f(2, f(3, z)))
```

Empieza **por la derecha** (por el final): combina el último con `z`, luego el penúltimo con
eso, etc. Por eso se llama *right*.

Su implementación es exactamente la plantilla de la sección 1:

In [ ]:
def foldRight[A, B](l: List[A])(nil: B)(cons: (A, B) => B): B =
    l match
        case Nil => nil
        case head :: tail => cons(head, foldRight(tail)(nil)(cons))

### Receta: de recursión a `foldRight` (traducción mecánica)

| En la versión recursiva | En `foldRight` |
|---|---|
| lo que devuelves en `case Nil => ...` | el **primer** argumento (`nil`) |
| el cuerpo de `case head :: tail => ...` | la **función** `(head, tailSol) => ...` |
| cada `miFuncion(tail)` | se sustituye por `tailSol` |

`collect` con `foldRight`:

In [ ]:
def collect[A, B](f: A => Option[B], l: List[A]): List[B] =
    l.foldRight(Nil: List[B])((head, tailSol) =>
        f(head) match
            case Some(b) => b :: tailSol
            case None    => tailSol
    )

O, con la sintaxis de Scala 3 que usa el profesor (el `:` al final de la línea significa
"lo que viene indentado debajo es el último argumento"):

In [ ]:
def collect[A, B](f: A => Option[B], l: List[A]): List[B] =
    l.foldRight(Nil: List[B]): (head, tailSol) =>
        f(head) match
            case Some(b) => b :: tailSol
            case None    => tailSol

### ¡Cuidado con el tipo del valor inicial!

Si escribes `l.foldRight(Nil)(...)`, Scala cree que el resultado es "lista de nada"
(`List[Nothing]`) y luego da error al hacer `b :: tailSol`. Dile el tipo:

- `Nil: List[B]` o `List[B]()`
- `(List[A](), List[B]())` para un par de listas
- `Right(Nil): Either[List[A], List[B]]`
- o bien da el tipo como parámetro: `l.foldRight[Either[List[A], List[B]]](Right(Nil))(...)`

### Usar `case` dentro de `foldRight`

Si necesitas pattern matching sobre `head` o sobre `tailSol`, en vez de
`(head, tailSol) => ...` escribe directamente los `case`. Cada caso recibe **el par**
`(head, tailSol)`:

```scala
l.foldRight(valorBase):
    case (Left(a), (as, bs))  => ...
    case (Right(b), (as, bs)) => ...
```

En Scala 2: `l.foldRight(valorBase){ case (...) => ...; case (...) => ... }`.

---

## 4. Ejemplos resueltos de exámenes

### `swap` (25-26 Móstoles)

Si todos son `Right`, devuelve `Right` con todos; si hay algún `Left`, devuelve `Left` con
todos los `Left`.

In [ ]:
// swap(List(Right('a'), Left(1), Right('c'), Left(2))) == Left(List(1, 2))
def swap[A, B](l: List[Either[A, B]]): Either[List[A], List[B]] =
    l match
        case Nil => Right(Nil)
        case head :: tail =>
            (head, swap(tail)) match        // ¡match sobre la cabeza Y la solución del resto a la vez!
                case (Right(b), Right(bs)) => Right(b :: bs)   // todo Right hasta ahora
                case (Left(a), Right(_))   => Left(List(a))    // primer Left: empiezo lista de As
                case (Left(a), Left(as))   => Left(a :: as)
                case (Right(_), Left(as))  => Left(as)         // ya había Left: ignoro el Right

def swapFR[A, B](l: List[Either[A, B]]): Either[List[A], List[B]] =
    l.foldRight[Either[List[A], List[B]]](Right(Nil)):
        case (Right(b), Right(bs)) => Right(b :: bs)
        case (Left(a), Right(_))   => Left(List(a))
        case (Left(a), Left(as))   => Left(a :: as)
        case (Right(_), Left(as))  => Left(as)

**Patrón muy útil:** `(head, f(tail)) match` y enumerar las combinaciones. Con `foldRight`
los `case` son exactamente los mismos.

### `split` de `Either` (25-26 Vicálvaro, 20-21 Extraordinaria)

In [ ]:
// split(List(Right('a'), Left(1), Right('c'), Left(2))) == (List(1, 2), List('a', 'c'))
def split[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    l match
        case Nil => (Nil, Nil)
        case head :: tail =>
            (head, split(tail)) match
                case (Left(a), (as, bs))  => (a :: as, bs)
                case (Right(b), (as, bs)) => (as, b :: bs)

def splitFR[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    l.foldRight((List[A](), List[B]())):
        case (Left(a), (as, bs))  => (a :: as, bs)
        case (Right(b), (as, bs)) => (as, b :: bs)

### `split` de pares (20-21 Extraordinaria, variante A)

In [ ]:
// split(List((1, true), (2, false))) == (List(1, 2), List(true, false))
def split[A, B](l: List[(A, B)]): (List[A], List[B]) =
    l match
        case Nil => (Nil, Nil)
        case (a, b) :: tail =>
            val (as, bs) = split(tail)        // desestructuro la solución del resto
            (a :: as, b :: bs)

### `intersect` con guardas (24-25 Móstoles)

In [ ]:
// intersect(List(4,6,5,7), List(3,4,5)) == List(4,5)
def intersect[A](l1: List[A], l2: List[A]): List[A] =
    l1 match
        case Nil => Nil
        case head :: tail if l2.contains(head) => head :: intersect(tail, l2)
        case _ :: tail => intersect(tail, l2)

def intersectFR[A](l1: List[A], l2: List[A]): List[A] =
    l1.foldRight(Nil: List[A]):
        case (head, tailSol) if l2.contains(head) => head :: tailSol
        case (_, tailSol) => tailSol

### `removeAll` y `distinct` (21-22 Extraordinaria)

In [ ]:
def removeAll[A](a: A, list: List[A]): List[A] =
    list.foldRight(List[A]()):
        case (`a`, tailSol)  => tailSol           // `a` entre comillas invertidas: "igual a a"
        case (head, tailSol) => head :: tailSol

// distinct: quita repetidos dejando la primera aparición
def distinct[A](list: List[A]): List[A] =
    list match
        case Nil => Nil
        case head :: tail => head :: distinct(removeAll(head, tail))

def distinctFR[A](list: List[A]): List[A] =
    list.foldRight(List[A]())((head, tailSol) => head :: removeAll(head, tailSol))

### `compress`: mirar dentro de la solución del resto (19-20 Diciembre)

In [ ]:
// compress(List('a','a','a','c','c','b')) == List(('a',3), ('c',2), ('b',1))
def compress[A](list: List[A]): List[(A, Int)] =
    list match
        case Nil => List()
        case head :: tail =>
            compress(tail) match
                case (`head`, n) :: rest => (head, n + 1) :: rest   // el resto empieza por el mismo: sumo 1
                case tailSol             => (head, 1) :: tailSol

def compressFR[A](list: List[A]): List[(A, Int)] =
    list.foldRight(List[(A, Int)]()):
        case (head, (h, n) :: rest) if h == head => (h, n + 1) :: rest
        case (head, tailSol) => (head, 1) :: tailSol

### `intersperse` (19-20 Julio)

In [ ]:
// intersperse(',')(List('h','o','l','a')) == List('h', ',', 'o', ',', 'l', ',', 'a')
def intersperse[A](a: A)(l: List[A]): List[A] =
    l match
        case Nil => Nil
        case head :: Nil => head :: Nil
        case head :: tail => head :: a :: intersperse(a)(tail)

def intersperseFR[A](a: A)(l: List[A]): List[A] =
    l.foldRight(Nil: List[A]):
        case (head, Nil)     => head :: Nil        // es el último: sin separador
        case (head, tailSol) => head :: a :: tailSol

### `sequence` de `Option` (20-21 Ordinaria)

Si todos son `Some`, devuelve `Some` con la lista; si hay algún `None`, `None`.

In [ ]:
def sequence[A](l: List[Option[A]]): Option[List[A]] =
    l.foldRight[Option[List[A]]](Some(Nil)):
        case (Some(h), Some(t)) => Some(h :: t)
        case _ => None

### Truco: devolver **una tupla** cuando `tailSol` no basta (`tails`, 23-24 Móstoles)

`tails(List(4,2,1))` debe dar `List(List(4,2,1), List(2,1), List(1), Nil)`.

Con `foldRight` la función solo recibe `head` y `tailSol`, **no** el resto de la lista
(`tail`). Solución: que la solución lleve **también** el resto, como un par, y al final
quedarte con lo que te interesa con `._1`:

In [ ]:
def tails[A](l: List[A]): List[List[A]] =
    l match
        case Nil => List(Nil)
        case _ :: t => l :: tails(t)

def tailsFR[A](l: List[A]): List[List[A]] =
    l.foldRight((List(List[A]()), List[A]())):       // (solución, lista reconstruida)
        case (h, (tailSol, tail)) => ((h :: tail) :: tailSol, h :: tail)
    ._1

Este truco ("lleva información extra en una tupla y al final extrae con `._1`") sirve para
muchos ejercicios. Otro ejemplo: `penultimate` (19-20 Prueba) lleva un `Boolean` de
"¿ya he pasado el último?".

### Avanzado: `foldRight` que devuelve **una función** (`lift`, 21-22 Móstoles)

`lift(l)(i)` devuelve `Some(elemento en la posición i)` o `None`.

In [ ]:
def lift[A](l: List[A]): Int => Option[A] =
    idx => l match
        case Nil => None
        case head :: tail => if idx == 0 then Some(head) else lift(tail)(idx - 1)

def liftFR[A](l: List[A]): Int => Option[A] =
    // para Nil: la función "siempre None".  liftTail: Int => Option[A] (solución del resto)
    l.foldRight((_: Int) => None: Option[A]): (head, liftTail) =>
        idx => if idx == 0 then Some(head) else liftTail(idx - 1)

Aquí `B` (el tipo de la solución) es `Int => Option[A]`. No te asustes: la receta es la
misma, solo que la "solución" es una función.

### Recursión sobre una estructura que contiene listas (22-23 Vicálvaro Especial)

In [ ]:
case class Struct[A](elements: List[Either[A, Struct[A]]])

def flatten[A](l: List[Either[A, Struct[A]]]): List[A] =
    l.foldRight(Nil: List[A]):
        case (Left(a), tailSol)          => a :: tailSol
        case (Right(Struct(h)), tailSol) => flatten(h) ++ tailSol   // recursión dentro

### Recursión sobre dos listas a la vez (`zip`, `map2`)

In [ ]:
def map2[A, B, C](as: List[A], bs: List[B])(f: (A, B) => C): List[C] =
    (as, bs) match
        case (a :: as2, b :: bs2) => f(a, b) :: map2(as2, bs2)(f)
        case _ => Nil              // alguna se acabó

---

## 5. Errores típicos

- **No poner el tipo del valor inicial** en `foldRight` (ver sección 3).
- **Confundir el orden de los argumentos.** En `foldRight` la función recibe
  `(head, tailSol)` — primero el elemento, luego la solución. En `foldLeft` es al revés
  (archivo 05).
- **Llamar recursivamente dentro de `foldRight`** a la propia función con `tail`: no hace
  falta, eso ya es `tailSol`. (Excepción: cuando la recursión es sobre **otra** cosa, como
  `flatten(h)` arriba.)
- **Casos en mal orden.** El caso más general (`case (head, tailSol) => ...`) va el último.
- **Olvidar el caso `Nil`** en la versión recursiva.

---

## 6. Ejercicios para practicar

### De exámenes reales (pide siempre versión recursiva **y** con `foldRight`)

| Función | Examen |
|---|---|
| `collect` | 25-26 Extraordinaria |
| `swap` | 25-26 Móstoles |
| `split` (Either) | 25-26 Vicálvaro, 20-21 Extraordinaria (variante B) |
| `intersect` | 24-25 Móstoles |
| `pair` | 23-24 Extraordinaria |
| `tails`, `paramorphism` | 23-24 Móstoles |
| `lift1st`, `lift2nd` | 22-23 Móstoles |
| `flatten` de `Struct` | 22-23 Vicálvaro Especial |
| `removeAll`, `distinct` | 21-22 Extraordinaria |
| `lift` | 21-22 Móstoles |
| `split` (pares) | 20-21 Extraordinaria (variante A) |
| `sequence` (Option y Either) | 20-21 Ordinaria |
| `compress` | 19-20 Diciembre |
| `intersperse` | 19-20 Julio |
| `penultimate` | 19-20 Prueba |

Más práctica: `PF-3/topic5/5.2 Recursion-exercises.ipynb`.

### Ejercicios nuevos (hazlos recursivos y con `foldRight`)

1. `length[A](l: List[A]): Int`
2. `myMap[A, B](l: List[A])(f: A => B): List[B]` (reimplementa `map`)
3. `myFilter[A](l: List[A])(p: A => Boolean): List[A]`
4. `countIf[A](l: List[A])(p: A => Boolean): Int` — cuántos cumplen `p`
5. `partition[A](l: List[A])(p: A => Boolean): (List[A], List[A])` — los que cumplen y los que no

<details>
<summary>Soluciones con foldRight</summary>

```scala
def length[A](l: List[A]): Int =
    l.foldRight(0)((_, tailSol) => 1 + tailSol)

def myMap[A, B](l: List[A])(f: A => B): List[B] =
    l.foldRight(Nil: List[B])((head, tailSol) => f(head) :: tailSol)

def myFilter[A](l: List[A])(p: A => Boolean): List[A] =
    l.foldRight(Nil: List[A])((head, tailSol) => if p(head) then head :: tailSol else tailSol)

def countIf[A](l: List[A])(p: A => Boolean): Int =
    l.foldRight(0)((head, tailSol) => if p(head) then tailSol + 1 else tailSol)

def partition[A](l: List[A])(p: A => Boolean): (List[A], List[A]) =
    l.foldRight((List[A](), List[A]())):
        case (head, (si, no)) => if p(head) then (head :: si, no) else (si, head :: no)
```

</details>

Siguiente: **05-recursion-final-y-foldLeft.ipynb**.